# LAB | Error Handling in Python

## Overview
This exercise notebook will help you practice error handling in Python using exceptions. You will write programs that handle various types of exceptions to ensure your code runs smoothly and handles errors gracefully.

In [ ]:
# --- Configuration ------------------------------------------------------------
# Plusieurs exercices demandent une saisie utilisateur. `input()` bloque, et leve
# une exception quand le notebook est execute sans terminal interactif
# (nbconvert, papermill, CI). `saisir()` utilise input() quand c'est possible et
# retombe sinon sur une liste d'entrees simulees : le notebook reste executable
# de bout en bout, et les exercices restent interactifs quand on les lance a la main.

def saisir(message, entrees_simulees=None):
    try:
        return input(message)
    except KeyboardInterrupt:
        raise                      # un vrai Ctrl-C doit passer (cf. exercice 7)
    except Exception:
        if entrees_simulees is None:
            raise
        valeur = next(entrees_simulees)
        print(f"{message}{valeur}    [entree simulee]")
        return valeur


print("Configuration chargee.")

Configuration chargee.


### Exercise 1: Handle ZeroDivisionError
Write a Python program to handle a `ZeroDivisionError` exception when dividing a number by zero.


In [2]:
# Exercice 1 : ZeroDivisionError
# On intercepte la division par zero et on rend une valeur exploitable
# plutot que de laisser le programme s'arreter.

def diviser(a, b):
    try:
        resultat = a / b
    except ZeroDivisionError:
        print(f"Erreur : division de {a} par zero impossible.")
        return None
    else:
        # `else` ne s'execute que si aucune exception n'a ete levee
        print(f"{a} / {b} = {resultat}")
        return resultat


diviser(10, 2)
diviser(10, 0)

# Nuance a connaitre : avec des entiers, // et % levent la meme exception,
# alors qu'avec des flottants, 10.0 / 0.0 leve ZeroDivisionError mais
# l'operation equivalente sous numpy renvoie inf avec un simple warning.
try:
    10 % 0
except ZeroDivisionError as e:
    print("Modulo par zero attrape aussi :", e)

10 / 2 = 5.0
Erreur : division de 10 par zero impossible.
Modulo par zero attrape aussi : integer modulo by zero



### Exercise 2: Raise ValueError for Invalid Input
Write a Python program that prompts the user to input an integer and raises a `ValueError` exception if the input is not a valid integer.



In [3]:
# Exercice 2 : lever une ValueError si la saisie n'est pas un entier
# `saisir()` (cellule de configuration) remplace input() quand le notebook
# est execute sans terminal interactif.

entrees_ex2 = iter(["42", "quarante-deux"])


def lire_entier(message="Entrez un entier : "):
    texte = saisir(message, entrees_ex2)
    try:
        return int(texte)
    except ValueError:
        # `from None` masque l'exception d'origine : le message reste lisible.
        raise ValueError(f"'{texte}' n'est pas un entier valide.") from None


for essai in range(2):
    try:
        valeur = lire_entier()
        print("Entier lu :", valeur)
    except ValueError as e:
        print("ValueError levee :", e)

ValueError levee : '' n'est pas un entier valide.
ValueError levee : '' n'est pas un entier valide.




### Exercise 3: Handle FileNotFoundError
Write a Python program that opens a file and handles a `FileNotFoundError` exception if the file does not exist.



In [4]:
# Exercice 3 : FileNotFoundError
import os
import tempfile

DOSSIER = tempfile.gettempdir()   # fichiers de test hors du depot


def lire_fichier(chemin):
    try:
        with open(chemin, "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        print(f"Fichier introuvable : {chemin}")
        return None


# Cas qui echoue
lire_fichier(os.path.join(DOSSIER, "fichier_qui_nexiste_pas.txt"))

# Cas qui fonctionne
chemin_ok = os.path.join(DOSSIER, "demo_ok.txt")
with open(chemin_ok, "w", encoding="utf-8") as f:
    f.write("contenu de test")

print("Contenu lu :", repr(lire_fichier(chemin_ok)))

# A noter : FileNotFoundError est une sous-classe de OSError. Attraper OSError
# fonctionnerait aussi, mais masquerait d'autres pannes (disque plein, droits...).
print("FileNotFoundError herite de OSError :", issubclass(FileNotFoundError, OSError))

Fichier introuvable : /var/folders/gh/t01w6z3100q8z4y2vlhw10c80000gn/T/fichier_qui_nexiste_pas.txt
Contenu lu : 'contenu de test'
FileNotFoundError herite de OSError : True




### Exercise 4: Raise TypeError for Non-Numerical Input
Write a Python program that prompts the user to input two numbers and raises a `TypeError` exception if the inputs are not numerical.



In [5]:
# Exercice 4 : lever une TypeError si les entrees ne sont pas numeriques
from numbers import Number


def additionner(a, b):
    for nom, valeur in (("a", a), ("b", b)):
        # bool est une sous-classe de int : on l'exclut explicitement.
        if isinstance(valeur, bool) or not isinstance(valeur, Number):
            raise TypeError(
                f"{nom} doit etre un nombre, recu {type(valeur).__name__} ({valeur!r})"
            )
    return a + b


cas = [(2, 3), (2.5, 4), ("2", 3), (2, None), (True, 3)]

for a, b in cas:
    try:
        print(f"{a!r} + {b!r} = {additionner(a, b)}")
    except TypeError as e:
        print("TypeError :", e)

2 + 3 = 5
2.5 + 4 = 6.5
TypeError : a doit etre un nombre, recu str ('2')
TypeError : b doit etre un nombre, recu NoneType (None)
TypeError : a doit etre un nombre, recu bool (True)




### Exercise 5: Handle PermissionError
Write a Python program that opens a file and handles a `PermissionError` exception if there is a permission issue.




In [6]:
# Exercice 5 : PermissionError
import os
import stat
import tempfile

chemin_protege = os.path.join(tempfile.gettempdir(), "fichier_protege.txt")

with open(chemin_protege, "w", encoding="utf-8") as f:
    f.write("secret")

os.chmod(chemin_protege, 0o000)   # plus aucun droit

try:
    with open(chemin_protege, "r", encoding="utf-8") as f:
        contenu = f.read()
    # Un compte root ignore les droits : dans ce cas aucune exception n'est levee.
    print("Lecture reussie malgre les droits retires (execution en root ?)")
except PermissionError as e:
    print("PermissionError attrapee :", e)
finally:
    # On restaure les droits pour pouvoir supprimer le fichier.
    os.chmod(chemin_protege, stat.S_IRUSR | stat.S_IWUSR)
    os.remove(chemin_protege)
    print("Fichier de test nettoye.")

PermissionError attrapee : [Errno 13] Permission denied: '/var/folders/gh/t01w6z3100q8z4y2vlhw10c80000gn/T/fichier_protege.txt'
Fichier de test nettoye.




### Exercise 6: Handle IndexError in List Operations
Write a Python program that executes an operation on a list and handles an `IndexError` exception if the index is out of range.




In [7]:
# Exercice 6 : IndexError

nombres = [10, 20, 30]


def element(liste, index):
    try:
        return liste[index]
    except IndexError:
        print(f"Index {index} hors limites (taille = {len(liste)})")
        return None


print("index 1  ->", element(nombres, 1))
print("index 10 ->", element(nombres, 10))
print("index -1 ->", element(nombres, -1))   # valide en Python : dernier element

# Piege classique : le slicing ne leve JAMAIS IndexError, il tronque en silence.
print("nombres[5:9] ->", nombres[5:9], "(aucune exception)")

index 1  -> 20
Index 10 hors limites (taille = 3)
index 10 -> None
index -1 -> 30
nombres[5:9] -> [] (aucune exception)




### Exercise 7: Handle KeyboardInterrupt Exception
Write a Python program that prompts the user to input a number and handles a `KeyboardInterrupt` exception if the user cancels the input.



In [ ]:
# Exercice 7 : KeyboardInterrupt
# KeyboardInterrupt herite de BaseException, PAS de Exception : un
# `except Exception` ne l'attrape donc pas. C'est voulu, pour qu'un Ctrl-C
# puisse toujours interrompre un programme.

entrees_ex7 = iter(["7"])


def demander_nombre():
    try:
        texte = saisir("Entrez un nombre (Ctrl-C pour annuler) : ", entrees_ex7)
        return float(texte)
    except KeyboardInterrupt:
        print("\nSaisie annulee par l'utilisateur.")
        return None
    except ValueError:
        print("Ce n'est pas un nombre.")
        return None


print("Resultat :", demander_nombre())

# Simulation d'un Ctrl-C, pour montrer que la branche fonctionne
try:
    raise KeyboardInterrupt
except KeyboardInterrupt:
    print("Ctrl-C simule : interruption attrapee proprement.")

print("Exception attrape KeyboardInterrupt ?", issubclass(KeyboardInterrupt, Exception))



### Exercise 8: Handle ArithmeticError
Write a Python program that executes division and handles an `ArithmeticError` exception if there is an arithmetic error.



In [ ]:
# Exercice 8 : ArithmeticError
# ArithmeticError est la classe parente de ZeroDivisionError, OverflowError
# et FloatingPointError : une seule clause couvre les trois.

import math


def calculer(operation, *args):
    try:
        return operation(*args)
    except ArithmeticError as e:
        print(f"{type(e).__name__} : {e}")
        return None


print("100 / 5   ->", calculer(lambda a, b: a / b, 100, 5))
print("100 / 0   ->", calculer(lambda a, b: a / b, 100, 0))
print("exp(1000) ->", calculer(math.exp, 1000))   # OverflowError

for classe in (ZeroDivisionError, OverflowError, FloatingPointError):
    print(f"{classe.__name__} herite d'ArithmeticError :",
          issubclass(classe, ArithmeticError))

100 / 5   -> 20.0
ZeroDivisionError : division by zero
100 / 0   -> None
OverflowError : math range error
exp(1000) -> None
ZeroDivisionError herite d'ArithmeticError : True
OverflowError herite d'ArithmeticError : True
FloatingPointError herite d'ArithmeticError : True




### Exercise 9: Handle UnicodeDecodeError
Write a Python program that opens a file and handles a `UnicodeDecodeError` exception if there is an encoding issue.



In [ ]:
# Exercice 9 : UnicodeDecodeError
import os
import tempfile

chemin_binaire = os.path.join(tempfile.gettempdir(), "fichier_binaire.dat")

# Octets invalides en UTF-8
with open(chemin_binaire, "wb") as f:
    f.write(b"\xff\xfe donnees binaires \x00\x81")

try:
    with open(chemin_binaire, "r", encoding="utf-8") as f:
        f.read()
except UnicodeDecodeError as e:
    print("UnicodeDecodeError :", e)
    print(f"   octet fautif a la position {e.start}, encodage attendu : {e.encoding}")

# Deux strategies de repli, selon ce qu'on veut faire du fichier :
with open(chemin_binaire, "r", encoding="utf-8", errors="replace") as f:
    print("\nAvec errors='replace' :", repr(f.read()))

with open(chemin_binaire, "rb") as f:
    print("En binaire (pas de decodage) :", f.read())

os.remove(chemin_binaire)

UnicodeDecodeError : 'utf-8' codec can't decode byte 0xff in position 0: invalid start byte
   octet fautif a la position 0, encodage attendu : utf-8

Avec errors='replace' : '�� donnees binaires \x00�'
En binaire (pas de decodage) : b'\xff\xfe donnees binaires \x00\x81'




### Exercise 10: Handle AttributeError
Write a Python program that executes an operation on an object and handles an `AttributeError` exception if the attribute does not exist.



In [ ]:
# Exercice 10 : AttributeError


class Patient:
    def __init__(self, nom):
        self.nom = nom


p = Patient("Dupont")

try:
    print(p.prenom)
except AttributeError as e:
    print("AttributeError :", e)

# Trois facons d'eviter le probleme plutot que de le rattraper :
print("getattr avec defaut :", getattr(p, "prenom", "(non renseigne)"))
print("hasattr             :", hasattr(p, "prenom"))
print("attributs existants :", list(vars(p)))

# Cas frequent en pratique : None qui traine dans une variable.
resultat = None
try:
    resultat.upper()
except AttributeError as e:
    print("Sur None ->", e)

AttributeError : 'Patient' object has no attribute 'prenom'
getattr avec defaut : (non renseigne)
hasattr             : False
attributs existants : ['nom']
Sur None -> 'NoneType' object has no attribute 'upper'




## Bonus Exercises

### Bonus Exercise 1: Handle Multiple Exceptions
Write a Python program that demonstrates handling multiple exceptions in one block.




In [ ]:
# Bonus 1 : gerer plusieurs exceptions dans un meme bloc

donnees = {"a": 1, "b": 0, "liste": [1, 2, 3]}


def operation(cas):
    try:
        if cas == "division":
            return donnees["a"] / donnees["b"]
        if cas == "cle":
            return donnees["inexistante"]
        if cas == "index":
            return donnees["liste"][99]
        if cas == "type":
            return donnees["a"] + "texte"
        return donnees["a"]
    except (ZeroDivisionError, KeyError, IndexError) as e:
        # Regroupees : meme traitement pour trois erreurs "de donnees"
        print(f"  erreur de donnees -> {type(e).__name__}: {e}")
    except TypeError as e:
        # Separee : traitement different
        print(f"  erreur de type -> {e}")
    except Exception as e:
        # Filet de securite en dernier : l'ordre compte, du plus precis au plus general
        print(f"  erreur inattendue -> {type(e).__name__}: {e}")
    finally:
        # S'execute toujours, exception ou non
        print(f"  [cas '{cas}' termine]")


for cas in ["division", "cle", "index", "type", "ok"]:
    print(f"Cas : {cas}")
    resultat = operation(cas)
    if resultat is not None:
        print("  resultat :", resultat)

Cas : division
  erreur de donnees -> ZeroDivisionError: division by zero
  [cas 'division' termine]
Cas : cle
  erreur de donnees -> KeyError: 'inexistante'
  [cas 'cle' termine]
Cas : index
  erreur de donnees -> IndexError: list index out of range
  [cas 'index' termine]
Cas : type
  erreur de type -> unsupported operand type(s) for +: 'int' and 'str'
  [cas 'type' termine]
Cas : ok
  [cas 'ok' termine]
  resultat : 1




### Bonus Exercise 2: Create Custom Exception
Create a custom exception class and raise it in your code when certain conditions are met.




In [ ]:
# Bonus 2 : exception personnalisee
# Une exception metier porte de l'information structuree, pas seulement un message :
# le code appelant peut alors reagir sans analyser une chaine de caracteres.


class ErreurDeSaisie(Exception):
    """Classe de base pour les erreurs de saisie de ce module."""


class ValeurHorsPlage(ErreurDeSaisie):
    def __init__(self, valeur, minimum, maximum):
        self.valeur = valeur
        self.minimum = minimum
        self.maximum = maximum
        super().__init__(
            f"{valeur} est hors de la plage autorisee [{minimum} - {maximum}]"
        )


def verifier_age(age):
    if not 0 <= age <= 120:
        raise ValeurHorsPlage(age, 0, 120)
    return age


for age in (35, 150):
    try:
        print("Age accepte :", verifier_age(age))
    except ValeurHorsPlage as e:
        print("ValeurHorsPlage :", e)
        # L'interet : les attributs sont exploitables directement
        print(f"   valeur recue = {e.valeur}, maximum attendu = {e.maximum}")

# Grace a l'heritage, un appelant peut attraper toute la famille d'un coup
try:
    verifier_age(-5)
except ErreurDeSaisie as e:
    print("Attrapee via la classe parente ErreurDeSaisie :", e)

Age accepte : 35
ValeurHorsPlage : 150 est hors de la plage autorisee [0 - 120]
   valeur recue = 150, maximum attendu = 120
Attrapee via la classe parente ErreurDeSaisie : -5 est hors de la plage autorisee [0 - 120]




### Bonus Exercise 3: Validate User Input with Exception Handling
Write a program that repeatedly prompts the user for valid input until they provide it, using exception handling to manage invalid inputs.



In [ ]:
# Bonus 3 : redemander tant que la saisie est invalide

entrees_ex_b3 = iter(["abc", "-4", "150", "35"])


def demander_age(max_essais=10):
    for essai in range(1, max_essais + 1):
        texte = saisir(f"[essai {essai}] Votre age : ", entrees_ex_b3)
        try:
            age = int(texte)
        except ValueError:
            print("   -> ce n'est pas un entier, recommencez.")
            continue
        try:
            return verifier_age(age)          # reutilise le Bonus 2
        except ValeurHorsPlage as e:
            print(f"   -> {e}")
    raise ErreurDeSaisie(f"abandon apres {max_essais} essais")


print("\nAge valide :", demander_age())

[essai 1] Votre age : abc    [entree simulee]
   -> ce n'est pas un entier, recommencez.
[essai 2] Votre age : -4    [entree simulee]
   -> -4 est hors de la plage autorisee [0 - 120]
[essai 3] Votre age : 150    [entree simulee]
   -> 150 est hors de la plage autorisee [0 - 120]
[essai 4] Votre age : 35    [entree simulee]

Age valide : 35




### Bonus Exercise 4: Log Errors to File
Modify your error handling to log errors to a text file instead of printing them to the console.



In [ ]:
# Bonus 4 : journaliser les erreurs dans un fichier au lieu de les afficher
import logging
import os
import tempfile

CHEMIN_LOG = os.path.join(tempfile.gettempdir(), "erreurs_lab.log")

logger = logging.getLogger("lab_erreurs")
logger.setLevel(logging.ERROR)
logger.handlers.clear()          # evite les doublons si la cellule est relancee

handler = logging.FileHandler(CHEMIN_LOG, mode="w", encoding="utf-8")
handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(handler)
logger.propagate = False         # ne pas reafficher dans la console


def diviser_avec_log(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        # exc_info=True ecrit la trace complete dans le fichier :
        # c'est ce qui rend un log exploitable apres coup.
        logger.error("Echec de %s / %s", a, b, exc_info=True)
        return None


print("10 / 2 =", diviser_avec_log(10, 2))
print("10 / 0 =", diviser_avec_log(10, 0))
print("1 / 0  =", diviser_avec_log(1, 0))

handler.flush()
print(f"\n--- contenu de {CHEMIN_LOG} ---")
print(open(CHEMIN_LOG, encoding="utf-8").read())

10 / 2 = 5.0
10 / 0 = None
1 / 0  = None

--- contenu de /sessions/rcw-0149xm93rba67bg1dd1b3g9g/tmp/erreurs_lab.log ---
2026-09-05 15:47:13,235 | ERROR | Echec de 10 / 0
Traceback (most recent call last):
  File "/sessions/rcw-0149xm93rba67bg1dd1b3g9g/tmp/ipykernel_7/3395100199.py", line 20, in diviser_avec_log
    return a / b
ZeroDivisionError: division by zero
2026-09-05 15:47:13,235 | ERROR | Echec de 1 / 0
Traceback (most recent call last):
  File "/sessions/rcw-0149xm93rba67bg1dd1b3g9g/tmp/ipykernel_7/3395100199.py", line 20, in diviser_avec_log
    return a / b
ZeroDivisionError: division by zero





### Bonus Exercise 5: Retry Logic on Exception
Implement retry logic for operations that could fail, allowing users to try again after encountering an error.



In [ ]:
# Bonus 5 : reessayer une operation qui peut echouer
import time


class ServiceInstable:
    """Simule un service qui echoue les 2 premieres fois, puis repond."""

    def __init__(self, echecs_avant_succes=2):
        self.appels = 0
        self.echecs_avant_succes = echecs_avant_succes

    def appeler(self):
        self.appels += 1
        if self.appels <= self.echecs_avant_succes:
            raise ConnectionError(f"service indisponible (appel {self.appels})")
        return f"reponse OK au bout de {self.appels} appels"


def avec_reessais(operation, essais=4, attente=0.1, exceptions=(ConnectionError,)):
    """Reessaie `operation`, avec une attente qui double a chaque echec.

    L'attente croissante evite de marteler un service deja en difficulte.
    Seules les exceptions listees sont reessayees : une erreur de programmation
    doit remonter immediatement, pas etre retentee 4 fois.
    """
    for tentative in range(1, essais + 1):
        try:
            return operation()
        except exceptions as e:
            if tentative == essais:
                print(f"Abandon apres {essais} tentatives.")
                raise
            delai = attente * (2 ** (tentative - 1))
            print(f"  tentative {tentative} echouee ({e}), nouvel essai dans {delai:.1f}s")
            time.sleep(delai)


print(avec_reessais(ServiceInstable(echecs_avant_succes=2).appeler))

print()
try:
    avec_reessais(ServiceInstable(echecs_avant_succes=99).appeler, essais=3)
except ConnectionError as e:
    print("Exception finalement propagee :", e)

  tentative 1 echouee (service indisponible (appel 1)), nouvel essai dans 0.1s
  tentative 2 echouee (service indisponible (appel 2)), nouvel essai dans 0.2s


reponse OK au bout de 3 appels

  tentative 1 echouee (service indisponible (appel 1)), nouvel essai dans 0.1s
  tentative 2 echouee (service indisponible (appel 2)), nouvel essai dans 0.2s


Abandon apres 3 tentatives.
Exception finalement propagee : service indisponible (appel 3)
